
# Bengali NLP Safety — V15.1 Fixed Rare-Label OOF Specialist

**Current Kaggle champion supplied to this notebook: V14 = 0.586.**

**Bug fix:** V15's first fold allocator could leave folds empty. V15.1 uses a global-balance grouped allocator and verifies all five folds before training.

This notebook does **not** replace the successful V14 prediction family blindly. It trains a new
BanglaBERT specialist for only four attribute labels:

- `implicit_toxicity`
- `contains_profanity`
- `threatening`
- `sexual_content`

The experiment uses **5-fold grouped out-of-fold (OOF) training** over the combined train + validation
labels. Repeated normalized texts are kept in the same fold. Each fold trains BanglaBERT with
moderate positive-class weighting, and epochs 3–4 are averaged.

The original 500-row validation set is then used only as a development comparison surface:
the new OOF probabilities are blended with the current V14-family probabilities and evaluated with
cross-fitted thresholds. A label is allowed to replace V14 only when it passes a conservative gate.

### What you should do

1. Open this notebook in Google Colab.
2. Set **Runtime → Change runtime type → T4 GPU**.
3. Run all cells from top to bottom.
4. At the end download **`v15_results.zip`**.
5. Send that ZIP back to ChatGPT.  
6. If the notebook prints a recommended CSV, **do not assume it beats 0.586 until Kaggle confirms it.**

No Kaggle API key or external CSV upload is required. The required V14/V5 data and probabilities are embedded.


In [ ]:
# 1. Settings — V15.1 fold-split bugfix
SEED = 42
N_FOLDS = 5
TARGET_LABELS = ["implicit_toxicity","contains_profanity","threatening","sexual_content"]
ALL_LABELS = ["is_toxic","implicit_toxicity","sarcasm","contains_profanity","threatening","identity_attack","sexual_content"]

MODEL_NAME = "csebuetnlp/banglabert"
MAX_LENGTH = 128
EPOCHS = 4
SNAPSHOT_EPOCHS = [3, 4]
LR = 1.5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
TRAIN_BATCH_SIZE = 8
ACCUM_STEPS = 2
EVAL_BATCH_SIZE = 32

# Conservative label replacement gate.
MIN_CROSSFIT_GAIN = 0.010
MIN_WINNING_FOLDS = 3
MAX_LOSING_FOLDS = 1

# Blend search. 0 means unchanged V14-family probability; 1 means pure new specialist.
BLEND_WEIGHTS = [i/20 for i in range(0, 21)]  # 0.00 ... 1.00
BLEND_MODES = ["prob", "logit"]

print("V15 target labels:", TARGET_LABELS)


In [ ]:

# 2. Install dependencies
import sys, subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "sentencepiece==0.2.1",
    "scikit-learn",
    "pandas",
    "numpy",
    "tqdm",
    "safetensors",
])
print("Installation complete.")


In [ ]:
import base64, pathlib
DATA_DIR = pathlib.Path('/content/v15_embedded'); DATA_DIR.mkdir(exist_ok=True)
PAYLOADS = {'train.csv': '...', 'validation.csv': '...', 'test.csv': '...', 'v2_valid.csv': '...', 'v2_test.csv': '...', 'validation_folds.csv': 'SUQsZm9sZApCSUNIXzAwMTIsMgpCSUNIXzAwMjMsMQpCSUNIXzAwMjYsMgpCSUNIXzAwMzIsNApCSUNIXzAwMzMsNApCSUNIXzAwNTksNApCSUNIXzAwNjAsMwpCSUNIXzAwNjEsMQpCSUNIXzAwNjMsMApCSUNIXzAwNzIsNApCSUNIXzAwNzYsMApCSUNIXzAwODAsMApCSUNIXzAxMTQsMQpCSUNIXzAxMTUsMApCSUNIXzAxMTgsNApCSUNIXzAxMjQsMApCSUNIXzAxMzAsMgpCSUNIXzAxMzcsMApCSUNIXzAxNTAsMApCSUNIXzAxNzMsMQpCSUNIXzAxODMsMwpCSUNIXzAxOTIsMApCSUNIXzAyMDAsMQpCSUNIXzAyMDMsMwpCSUNIXzAyMDYsMgpCSUNIXzAyNDUsMQpCSUNIXzAyNTYsMQpCSUNIXzAyNTcsMwpCSUNIXzAyNjIsMApCSUNIXzAyNzQsNApCSUNIXzAyNzUsNApCSUNIXzAyODQsMwpCSUNIXzAyODUsMgpCSUNIXzAyODcsMApCSUNIXzAyOTAsMgpCSUNIXzAzMDAsMwpCSUNIXzAzMTEsMApCSUNIXzAzMTcsMQpCSUNIXzAzMjAsMgpCSUNIXzAzNTYsMApCSUNIXzAzNjIsNApCSUNIXzAzODAsMwpCSUNIXzAzOTgsMQpCSUNIXzA0MDEsMwpCSUNIXzA0MDMsMQpCSUNIXzA0MDksMApCSUNIXzA0MTksMgpCSUNIXzA0NTgsMwpCSUNIXzA0NjgsMQpCSUNIXzA0OTMsMQpCSUNIXzA1MDAsMwpCSUNIXzA1MDEsMQpCSUNIXzA1MTksMQpCSUNIXzA1MzUsMQpCSUNIXzA1NTUsMwpCSUNIXzA1NzUsMgpCSUNIXzA1NzYsNApCSUNIXzA1ODUsMgpCSUNIXzA1OTYsNApCSUNIXzA1OTcsMQpCSUNIXzA2MjEsMApCSUNIXzA2NTIsMwpCSUNIXzA2NTgsMgpCSUNIXzA2NzMsMgpCSUNIXzA2OTksMQpCSUNIXzA3MTQsMApCSUNIXzA3MjAsMwpCSUNIXzA3MzIsNApCSUNIXzA3NTUsMApCSUNIXzA3ODIsMgpCSUNIXzA3ODMsMQpCSUNIXzA4MDQsMApCSUNIXzA4MTIsNApCSUNIXzA4MTgsMQpCSUNIXzA4MjEsMQpCSUNIXzA4MzAsNApCSUNIXzA4NTIsNApCSUNIXzA4NTQsNApCSUNIXzA4NzUsMgpCSUNIXzA4ODIsMgpCSUNIXzA4OTUsMApCSUNIXzA5MTIsMgpCSUNIXzA5MTcsMwpCSUNIXzA5MjIsMQpCSUNIXzA5MjQsMgpCSUNIXzA5MjYsNApCSUNIXzA5MjcsMwpCSUNIXzA5MzUsMQpCSUNIXzA5MzcsMApCSUNIXzA5NTQsNApCSUNIXzA5NTksMQpCSUNIXzA5NjUsMwpCSUNIXzA5NzIsNApCSUNIXzA5NzgsMgpCSUNIXzA5OTQsMgpCSUNIXzA5OTUsMApCSUNIXzA5OTksMQpCSUNIXzEwMDEsMApCSUNIXzEwMDUsMApCSUNIXzEwMDgsMApCSUNIXzEwMjYsMApCSUNIXzEwMzIsMgpCSUNIXzEwNDQsMQpCSUNIXzEwNTUsNApCSUNIXzEwNTcsMwpCSUNIXzEwNTgsMgpCSUNIXzEwNjksMgpCSUNIXzEwNzUsNApCSUNIXzEwODIsMQpCSUNIXzEwOTYsNApCSUNIXzEwOTcsMwpCSUNIXzExMDIsMQpCSUNIXzExMDUsMQpCSUNIXzExMDcsNApCSUNIXzExMTksMgpCSUNIXzExNDMsMwpCSUNIXzExNTcsMgpCSUNIXzExNjgsMQpCSUNIXzExNzIsMQpCSUNIXzExODAsMQpCSUNIXzExOTUsMQpCSUNIXzExOTksMApCSUNIXzEyMDIsMgpCSUNIXzEyMTcsMApCSUNIXzEyMzIsMwpCSUNIXzEyMzcsMgpCSUNIXzEyNDMsMgpCSUNIXzEyNDcsMApCSUNIXzEyNjQsMwpCSUNIXzEyNjgsMApCSUNIXzEyOTAsMwpCSUNIXzEyOTQsMApCSUNIXzEzMDgsMgpCSUNIXzEzMzIsMApCSUNIXzEzMzUsMQpCSUNIXzEzNjAsMwpCSUNIXzEzNjksMApCSUNIXzEzNzMsMwpCSUNIXzEzODQsNApCSUNIXzEzOTYsMApCSUNIXzE0MDMsMQpCSUNIXzE0MDgsNApCSUNIXzE0MTIsMQpCSUNIXzE0MTcsMgpCSUNIXzE0MTgsMwpCSUNIXzE0MjUsMwpCSUNIXzE0MjksMQpCSUNIXzE0NTQsMgpCSUNIXzE0NzEsMQpCSUNIXzE0NzYsMgpCSUNIXzE0ODMsMQpCSUNIXzE0OTAsNApCSUNIXzE0OTEsMwpCSUNIXzE0OTYsMQpCSUNIXzE1MDMsMApCSUNIXzE1MjgsMQpCSUNIXzE1MjksMgpCSUNIXzE1MzQsNApCSUNIXzE1MzksMwpCSUNIXzE1NDAsNApCSUNIXzE1NDMsMQpCSUNIXzE1NTYsMApCSUNIXzE1NjMsNApCSUNIXzE1NzYsNApCSUNIXzE1ODIsMApCSUNIXzE1OTYsMwpCSUNIXzE2MDgsMQpCSUNIXzE2MTYsMgpCSUNIXzE2MTcsNApCSUNIXzE2MjQsMApCSUNIXzE2MjUsNApCSUNIXzE2MzMsMwpCSUNIXzE2NDEsMwpCSUNIXzE2NDgsMwpCSUNIXzE2NjAsMQpCSUNIXzE2NjMsMwpCSUNIXzE2NjksMApCSUNIXzE2OTAsNApCSUNIXzE2OTUsMQpCSUNIXzE3MzYsNApCSUNIXzE3MzksMgpCSUNIXzE3NDUsNApCSUNIXzE3NTgsMwpCSUNIXzE3NzUsMApCSUNIXzE3ODQsNApCSUNIXzE4MjEsMApCSUNIXzE4MjMsMwpCSUNIXzE4MzksMQpCSUNIXzE4NjAsMwpCSUNIXzE4NjEsMwpCSUNIXzE4NjIsMApCSUNIXzE4ODQsMQpCSUNIXzE5MDYsMApCSUNIXzE5MTIsMwpCSUNIXzE5MjYsMQpCSUNIXzE5NjQsMgpCSUNIXzE5NjUsNApCSUNIXzE5ODcsMgpCSUNIXzE5ODgsMgpCSUNIXzE5OTAsMApCSUNIXzIwMDUsNApCSUNIXzIwMjQsNApCSUNIXzIwMjcsMApCSUNIXzIwMzEsMgpCSUNIXzIwNTAsMwpCSUNIXzIwNTksNApCSUNIXzIwNjIsMQpCSUNIXzIwNjgsMwpCSUNIXzIwNjksMQpCSUNIXzIwNzIsMQpCSUNIXzIwOTcsMgpCSUNIXzIxMDAsMApCSUNIXzIxMDYsMQpCSUNIXzIxMTMsMwpCSUNIXzIxMTUsNApCSUNIXzIxMjcsMApCSUNIXzIxMjksMApCSUNIXzIxMzgsNApCSUNIXzIxNTEsMApCSUNIXzIxNTMsMQpCSUNIXzIxNjEsMQpCSUNIXzIxNzAsMApCSUNIXzIxODIsNApCSUNIXzIxODUsMApCSUNIXzIxODgsNApCSUNIXzIyMTQsMwpCSUNIXzIyNTcsMQpCSUNIXzIyNzIsMgpCSUNIXzIyOTMsNApCSUNIXzIzMDQsMQpCSUNIXzIzMDYsMgpCSUNIXzIzMTQsMwpCSUNIXzIzMzksMgpCSUNIXzIzNDMsMApCSUNIXzIzNTksMgpCSUNIXzIzNjgsMQpCSUNIXzIzNzcsMApCSUNIXzIzODQsMwpCSUNIXzIzODYsMApCSUNIXzIzOTAsMwpCSUNIXzI0MDYsMwpCSUNIXzI0MTQsNApCSUNIXzI0NTYsMwpCSUNIXzI0NjQsNApCSUNIXzI0NzYsNApCSUNIXzI0NzcsMgpCSUNIXzI0NzgsMgpCSUNIXzI0ODEsNApCSUNIXzI0ODksNApCSUNIXzI0OTAsNApCSUNIXzI0OTEsMApCSUNIXzI1MTksMQpCSUNIXzI1MjMsMApCSUNIXzI1MjksMgpCSUNIXzI1MzQsMgpCSUNIXzI1NDgsMgpCSUNIXzI1NDksMgpCSUNIXzI1NjAsMwpCSUNIXzI1NjQsMwpCSUNIXzI1NzgsMApCSUNIXzI1ODQsNApCSUNIXzI1OTgsMwpCSUNIXzI2MDIsMApCSUNIXzI2MjIsMApCSUNIXzI2NDEsMgpCSUNIXzI2NDQsMgpCSUNIXzI2NjUsNApCSUNIXzI2NzYsMApCSUNIXzI2NzksMApCSUNIXzI2ODksMwpCSUNIXzI2OTMsMwpCSUNIXzI2OTQsMApCSUNIXzI3MjIsNApCSUNIXzI3MzQsNApCSUNIXzI3MzUsMQpCSUNIXzI3MzcsMgpCSUNIXzI3NjEsMApCSUNIXzI4MDgsMApCSUNIXzI4MTMsMwpCSUNIXzI4MTUsMQpCSUNIXzI4MzAsNApCSUNIXzI4MzcsNApCSUNIXzI4MzksMgpCSUNIXzI4NDIsMQpCSUNIXzI4NTgsMgpCSUNIXzI4NzIsMQpCSUNIXzI4NzQsMwpCSUNIXzI4NzgsMQpCSUNIXzI4ODIsMwpCSUNIXzI4OTUsMwpCSUNIXzI4OTcsNApCSUNIXzI5MDUsMwpCSUNIXzI5MTAsMApCSUNIXzI5MjYsMQpCSUNIXzI5MzUsNApCSUNIXzI5MzcsMgpCSUNIXzI5NDUsMQpCSUNIXzI5NDcsMwpCSUNIXzI5NDgsMgpCSUNIXzI5NjAsMwpCSUNIXzI5NjEsMwpCSUNIXzI5NjUsNApCSUNIXzI5NjYsMApCSUNIXzI5NjcsMwpCSUNIXzI5ODYsNApCSUNIXzI5OTUsMwpCSUNIXzI5OTgsMQpCSUNIXzMwMzIsNApCSUNIXzMwMzgsMApCSUNIXzMwNDEsMQpCSUNIXzMwNTMsMwpCSUNIXzMwNTYsMQpCSUNIXzMwNjIsMQpCSUNIXzMwNzAsMgpCSUNIXzMwNzUsMgpCSUNIXzMwNzgsNApCSUNIXzMwOTAsMgpCSUNIXzMxMTMsMgpCSUNIXzMxMzQsNApCSUNIXzMxMzgsNApCSUNIXzMxNTMsMgpCSUNIXzMxNzMsMwpCSUNIXzMxOTEsNApCSUNIXzMyMDEsMgpCSUNIXzMyMDQsMgpCSUNIXzMyMzYsMwpCSUNIXzMyMzgsMwpCSUNIXzMyNDIsMQpCSUNIXzMyNDUsMApCSUNIXzMyNDgsMApCSUNIXzMyNTQsNApCSUNIXzMyNjQsMApCSUNIXzMyNjUsMQpCSUNIXzMyNzIsMApCSUNIXzMyNzUsMApCSUNIXzMyNzksMApCSUNIXzMyODMsNApCSUNIXzMyOTYsMwpCSUNIXzMzMDEsMQpCSUNIXzMzMjAsMQpCSUNIXzMzNDMsMwpCSUNIXzMzNTAsNApCSUNIXzMzNjEsMwpCSUNIXzMzNzIsMApCSUNIXzMzNzUsMApCSUNIXzMzNzgsNApCSUNIXzMzODAsNApCSUNIXzM0MDksMApCSUNIXzM0MTYsMApCSUNIXzM0MjUsMApCSUNIXzM0MjYsMwpCSUNIXzM0MzMsMwpCSUNIXzM0NDAsMwpCSUNIXzM0NzgsMApCSUNIXzM0ODQsNApCSUNIXzM0ODcsMQpCSUNIXzM0ODgsMQpCSUNIXzM0OTQsMgpCSUNIXzM0OTYsMApCSUNIXzM1MDAsMApCSUNIXzM1MDQsMApCSUNIXzM1MjEsNApCSUNIXzM1MzUsNApCSUNIXzM1MzYsMwpCSUNIXzM1MzcsMApCSUNIXzM1NTEsMwpCSUNIXzM1NzUsMQpCSUNIXzM1NzksNApCSUNIXzM1OTUsNApCSUNIXzM1OTcsMwpCSUNIXzM2MDEsNApCSUNIXzM2MjcsNApCSUNIXzM2MjksMwpCSUNIXzM2MzEsMwpCSUNIXzM2NDcsMQpCSUNIXzM2NTMsMQpCSUNIXzM2NjYsMwpCSUNIXzM2NzUsMwpCSUNIXzM2ODAsNApCSUNIXzM2OTQsMwpCSUNIXzM2OTcsMwpCSUNIXzM3MTIsMApCSUNIXzM3MTksMgpCSUNIXzM3MjksMwpCSUNIXzM3NDMsMQpCSUNIXzM3NTMsMgpCSUNIXzM3NjAsMApCSUNIXzM3NjksMwpCSUNIXzM3NzIsMApCSUNIXzM3NzMsMwpCSUNIXzM3ODMsMQpCSUNIXzM3OTUsMgpCSUNIXzM3OTYsMApCSUNIXzM4MTMsNApCSUNIXzM4MjYsMwpCSUNIXzM4NDMsMgpCSUNIXzM4NTgsNApCSUNIXzM4ODMsMgpCSUNIXzM4ODksMQpCSUNIXzM4OTYsMQpCSUNIXzM5MDAsMwpCSUNIXzM5MTAsMQpCSUNIXzM5MzYsMApCSUNIXzM5MzksMgpCSUNIXzM5NDgsMApCSUNIXzM5NTIsMwpCSUNIXzM5NTMsNApCSUNIXzM5NTQsMApCSUNIXzM5NjUsNApCSUNIXzM5NjYsNApCSUNIXzM5NzAsMgpCSUNIXzM5ODIsMgpCSUNIXzM5OTYsMQpCSUNIXzQwMTYsMwpCSUNIXzQwMTgsMwpCSUNIXzQwMTksMQpCSUNIXzQwMjksMgpCSUNIXzQwNDksMgpCSUNIXzQwNTIsNApCSUNIXzQwNjQsMwpCSUNIXzQwNzQsMQpCSUNIXzQwNzYsMwpCSUNIXzQwNzgsMApCSUNIXzQwODQsMApCSUNIXzQwODksMwpCSUNIXzQxMDMsMwpCSUNIXzQxMTIsNApCSUNIXzQxMjEsMwpCSUNIXzQxNDIsMgpCSUNIXzQxNDMsMQpCSUNIXzQxNDQsMwpCSUNIXzQxNDgsMgpCSUNIXzQxNzIsMgpCSUNIXzQxNzcsMgpCSUNIXzQxNzgsMgpCSUNIXzQxODgsMApCSUNIXzQxOTAsMQpCSUNIXzQxOTUsMgpCSUNIXzQyMTQsMgpCSUNIXzQyMTksMApCSUNIXzQyMjcsMgpCSUNIXzQyNDcsNApCSUNIXzQyNTQsMApCSUNIXzQyNjEsNApCSUNIXzQyNjUsMwpCSUNIXzQyNjksMApCSUNIXzQyNzYsNApCSUNIXzQyNzgsMgpCSUNIXzQyODEsMgpCSUNIXzQyODYsMgpCSUNIXzQzMDgsNApCSUNIXzQzMTAsMQpCSUNIXzQzMzcsMApCSUNIXzQzNDcsMgpCSUNIXzQzNTUsMgpCSUNIXzQzNzEsMQpCSUNIXzQ0MDMsNApCSUNIXzQ0MDcsMQpCSUNIXzQ0MDksMApCSUNIXzQ0MTUsMApCSUNIXzQ0MTcsMQpCSUNIXzQ0MjAsMApCSUNIXzQ0MjIsNApCSUNIXzQ0MjYsMQpCSUNIXzQ0MzIsNApCSUNIXzQ0NDAsMQpCSUNIXzQ0NDUsMApCSUNIXzQ0NTksNApCSUNIXzQ0NjAsNApCSUNIXzQ0NjQsMwpCSUNIXzQ0NjksMQpCSUNIXzQ0NzEsMgpCSUNIXzQ0ODUsMApCSUNIXzQ0OTAsMwpCSUNIXzQ1MDEsMQpCSUNIXzQ1MTEsMwpCSUNIXzQ1MTcsMgpCSUNIXzQ1MjUsMApCSUNIXzQ1MjcsMApCSUNIXzQ1MzEsMQpCSUNIXzQ1MzksMgpCSUNIXzQ1NDMsNApCSUNIXzQ1NjQsMQpCSUNIXzQ1ODIsMApCSUNIXzQ1ODMsMQpCSUNIXzQ2MDksMQpCSUNIXzQ2MTEsMApCSUNIXzQ2MzgsMgpCSUNIXzQ2NDgsMQpCSUNIXzQ2NzMsMgpCSUNIXzQ2NzgsMgpCSUNIXzQ2ODYsMgpCSUNIXzQ2OTcsNApCSUNIXzQ3MjIsMQpCSUNIXzQ3MjYsNApCSUNIXzQ3MzcsMApCSUNIXzQ3NDYsMQpCSUNIXzQ3NjIsMwpCSUNIXzQ3NzAsMQpCSUNIXzQ3OTIsMgo=', 'banglabert_predictions.npz': '...', 'xlmr_predictions.npz': '...', 'submission_v14_champion.csv': '...'}

for name, data in PAYLOADS.items():
    (DATA_DIR/name).write_bytes(base64.b64decode(data))
print("Restored embedded files:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ", p.name, p.stat().st_size, "bytes")


In [ ]:

# 4. Imports, reproducibility, and data
import os, re, gc, math, random, time, json, zipfile, hashlib, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, precision_recall_curve
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
assert torch.cuda.is_available(), "Enable a GPU in Colab: Runtime → Change runtime type → T4 GPU"

train = pd.read_csv(DATA_DIR/"train.csv", dtype={"ID":str})
valid = pd.read_csv(DATA_DIR/"validation.csv", dtype={"ID":str})
test = pd.read_csv(DATA_DIR/"test.csv", dtype={"ID":str})
v2_valid = pd.read_csv(DATA_DIR/"v2_valid.csv", dtype={"ID":str})
v2_test = pd.read_csv(DATA_DIR/"v2_test.csv", dtype={"ID":str})
valid_folds = pd.read_csv(DATA_DIR/"validation_folds.csv", dtype={"ID":str})
champion = pd.read_csv(DATA_DIR/"submission_v14_champion.csv", dtype={"ID":str})

with np.load(DATA_DIR/"banglabert_predictions.npz") as z:
    old_bang_valid = z["valid"].copy()
    old_bang_test = z["test"].copy()
with np.load(DATA_DIR/"xlmr_predictions.npz") as z:
    old_xlmr_valid = z["valid"].copy()
    old_xlmr_test = z["test"].copy()

assert len(train)==3297 and len(valid)==500 and len(test)==1000
assert champion["ID"].tolist() == test["ID"].tolist()
assert v2_valid["ID"].tolist() == valid["ID"].tolist()
assert v2_test["ID"].tolist() == test["ID"].tolist()

combined = pd.concat([train, valid], ignore_index=True)
print("train/valid/combined/test:", len(train), len(valid), len(combined), len(test))
print("Target positives in combined:")
display(combined[TARGET_LABELS].sum().to_frame("positives"))


In [ ]:
# 5. Conservative text normalization and GUARANTEED balanced grouped 5-fold assignment
def normalize_group_text(x):
    x = unicodedata.normalize("NFKC", str(x))
    x = re.sub(r"\s+", " ", x).strip().lower()
    return x

combined["group_text"] = combined["text"].map(normalize_group_text)
test["clean_text"] = test["text"].fillna("").astype(str)
combined["clean_text"] = combined["text"].fillna("").astype(str)

# One normalized-text group must stay entirely inside one fold.
group_stats = combined.groupby("group_text", sort=False).agg(
    size=("ID","size"),
    **{f"sum_{l}":(l,"sum") for l in TARGET_LABELS}
).reset_index()

global_pos = np.array([combined[l].sum() for l in TARGET_LABELS], dtype=float)
total_rows = float(len(combined))

# Prioritize groups carrying rare positives, then use a GLOBAL imbalance cost.
# This fixes V15's bug where minimizing each candidate fold independently could
# fill only 3 folds and leave folds 3/4 empty.
rarity = 1.0 / np.sqrt(np.maximum(global_pos, 1.0))
group_label_matrix = group_stats[[f"sum_{l}" for l in TARGET_LABELS]].to_numpy(float)
priority = (group_label_matrix * rarity).sum(axis=1) + 0.005 * group_stats["size"].to_numpy()

rng = np.random.default_rng(SEED)
jitter = rng.random(len(group_stats)) * 1e-6
order = np.argsort(-(priority + jitter))

fold_pos = np.zeros((N_FOLDS, len(TARGET_LABELS)), dtype=float)
fold_size = np.zeros(N_FOLDS, dtype=float)
assignment = {}

for idx in order:
    row = group_stats.iloc[idx]
    vec = np.array([row[f"sum_{l}"] for l in TARGET_LABELS], dtype=float)
    size = float(row["size"])

    costs = []
    for f in range(N_FOLDS):
        pos_after = fold_pos.copy()
        size_after = fold_size.copy()
        pos_after[f] += vec
        size_after[f] += size

        # Ideal share for every fold is 1/N_FOLDS of each label and total rows.
        label_share = pos_after / np.maximum(global_pos[None, :], 1.0)
        size_share = size_after / total_rows

        label_cost = np.mean((label_share - 1.0/N_FOLDS) ** 2)
        size_cost = np.mean((size_share - 1.0/N_FOLDS) ** 2)
        costs.append(label_cost + 0.35 * size_cost)

    min_cost = min(costs)
    tied = np.flatnonzero(np.isclose(costs, min_cost, rtol=0, atol=1e-15))
    best = int(rng.choice(tied))

    assignment[row["group_text"]] = best
    fold_pos[best] += vec
    fold_size[best] += size

combined["fold"] = combined["group_text"].map(assignment).astype(int)

# HARD SAFETY CHECKS — fail here, before any GPU training.
assert combined["fold"].between(0, N_FOLDS-1).all()
assert combined.groupby("group_text")["fold"].nunique().max() == 1, "A duplicate-text group crossed folds."

fold_counts = combined.groupby("fold").size().reindex(range(N_FOLDS), fill_value=0)
target_fold_size = len(combined) / N_FOLDS

assert (fold_counts > 0).all(), f"Empty fold detected: {fold_counts.to_dict()}"
assert fold_counts.min() >= 0.80 * target_fold_size, f"Fold too small: {fold_counts.to_dict()}"
assert fold_counts.max() <= 1.20 * target_fold_size, f"Fold too large: {fold_counts.to_dict()}"

positive_table = combined.groupby("fold")[TARGET_LABELS].sum().reindex(range(N_FOLDS), fill_value=0)
assert (positive_table.sum(axis=0).to_numpy() == combined[TARGET_LABELS].sum().to_numpy()).all()
# Every target has enough positives to appear in every fold for this dataset.
assert (positive_table > 0).all().all(), f"A target is absent from a fold:\n{positive_table}"

print("✅ Fold construction passed all safety checks.")
print("Fold sizes:", fold_counts.to_dict())
print("\nPositive counts by fold:")
display(positive_table)


In [ ]:
# 6. Dataset helpers
class TextDataset(Dataset):
    def __init__(self, frame, tokenizer, with_labels=True):
        self.texts = frame["clean_text"].tolist()
        self.targets = frame[TARGET_LABELS].to_numpy(np.float32) if with_labels else None
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        enc = self.tokenizer(
            self.texts[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        if self.targets is not None:
            enc["labels"] = self.targets[i]
        return enc

class Collator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
    def __call__(self, items):
        labels = None
        if "labels" in items[0]:
            labels = torch.tensor(np.stack([x.pop("labels") for x in items]), dtype=torch.float32)
        batch = self.tokenizer.pad(items, padding=True, pad_to_multiple_of=8, return_tensors="pt")
        if labels is not None:
            batch["labels"] = labels
        return batch

@torch.no_grad()
def predict(model, loader):
    if len(loader.dataset) == 0:
        raise ValueError("Prediction dataset is empty. Fold construction should have caught this before training.")
    model.eval()
    out = []
    for batch in loader:
        batch.pop("labels", None)
        inputs = {k:v.to("cuda") for k,v in batch.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(**inputs).logits
        out.append(torch.sigmoid(logits.float()).cpu().numpy())
    if not out:
        raise ValueError("Prediction loader produced zero batches.")
    return np.concatenate(out)


In [ ]:
# 7. Train 5 OOF specialist folds — with per-fold cache and safety checks
RUN_DIR = Path("/content/v15_output")
RUN_DIR.mkdir(exist_ok=True)

oof = np.zeros((len(combined), len(TARGET_LABELS)), dtype=np.float32)
test_fold_probs = []
histories = []

fold_seeds = [42, 3407, 2026, 17, 1337]

for fold in range(N_FOLDS):
    print("\n" + "="*70)
    print("FOLD", fold)
    seed_everything(fold_seeds[fold])

    tr = combined[combined["fold"] != fold].copy()
    va = combined[combined["fold"] == fold].copy()

    # Safety checks before allocating/downloading a model.
    assert len(tr) > 0, f"Fold {fold}: empty training split"
    assert len(va) > 0, f"Fold {fold}: empty validation split"
    assert set(tr["group_text"]).isdisjoint(set(va["group_text"])), f"Fold {fold}: group leakage"
    print(f"train rows={len(tr):,} | validation rows={len(va):,}")
    print("validation positives:", va[TARGET_LABELS].sum().to_dict())

    cache_path = RUN_DIR / f"fold_{fold}_predictions.npz"
    history_path = RUN_DIR / f"fold_{fold}_history.csv"

    # Reuse a successfully completed fold if this cell is rerun in the same Colab runtime.
    if cache_path.exists():
        with np.load(cache_path, allow_pickle=False) as z:
            cached_idx = z["va_index"]
            cached_valid = z["valid"]
            cached_test = z["test"]
        expected_idx = va.index.to_numpy()
        if (
            np.array_equal(cached_idx, expected_idx)
            and cached_valid.shape == (len(va), len(TARGET_LABELS))
            and cached_test.shape == (len(test), len(TARGET_LABELS))
            and np.isfinite(cached_valid).all()
            and np.isfinite(cached_test).all()
        ):
            print("✅ Reusing completed fold cache:", cache_path.name)
            oof[expected_idx] = cached_valid
            test_fold_probs.append(cached_test)
            if history_path.exists():
                histories.extend(pd.read_csv(history_path).to_dict("records"))
            continue
        else:
            print("Ignoring incompatible fold cache:", cache_path.name)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    collator = Collator(tokenizer)
    tr_loader = DataLoader(
        TextDataset(tr, tokenizer, True),
        batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=0,
        pin_memory=True, collate_fn=collator
    )
    va_loader = DataLoader(
        TextDataset(va, tokenizer, True),
        batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0,
        pin_memory=True, collate_fn=collator
    )
    te_loader = DataLoader(
        TextDataset(test, tokenizer, False),
        batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0,
        pin_memory=True, collate_fn=collator
    )

    assert len(tr_loader) > 0 and len(va_loader) > 0 and len(te_loader) > 0

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(TARGET_LABELS),
        problem_type="multi_label_classification"
    ).to("cuda")

    positives = tr[TARGET_LABELS].sum().to_numpy(float)
    negatives = len(tr) - positives
    pos_weight = np.sqrt(negatives / np.maximum(positives,1))
    pos_weight = np.clip(pos_weight, 1.0, 4.0)
    pos_weight_t = torch.tensor(pos_weight, dtype=torch.float32, device="cuda")
    print("pos_weight:", dict(zip(TARGET_LABELS, np.round(pos_weight,3))))

    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_t)
    no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]
    grouped = [
        {"params":[p for n,p in model.named_parameters() if p.requires_grad and not any(x in n for x in no_decay)],
         "weight_decay":WEIGHT_DECAY},
        {"params":[p for n,p in model.named_parameters() if p.requires_grad and any(x in n for x in no_decay)],
         "weight_decay":0.0},
    ]
    optimizer = torch.optim.AdamW(grouped, lr=LR)

    update_steps_per_epoch = math.ceil(len(tr_loader)/ACCUM_STEPS)
    total_steps = update_steps_per_epoch * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps*WARMUP_RATIO),
        num_training_steps=total_steps
    )
    scaler = torch.amp.GradScaler("cuda")
    snapshots_valid, snapshots_test = [], []
    fold_history = []

    for epoch in range(1, EPOCHS+1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = 0.0
        seen = 0

        for step, batch in enumerate(tqdm(tr_loader, desc=f"fold {fold} epoch {epoch}/{EPOCHS}")):
            target = batch.pop("labels").to("cuda")
            inputs = {k:v.to("cuda") for k,v in batch.items()}

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(**inputs).logits

            loss = loss_fn(logits.float(), target)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite training loss")

            scaler.scale(loss/ACCUM_STEPS).backward()
            running += float(loss.item()) * len(target)
            seen += len(target)

            if (step+1) % ACCUM_STEPS == 0 or step+1 == len(tr_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= old_scale:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        rec = {"fold":fold, "epoch":epoch, "loss":running/max(seen,1)}

        if epoch in SNAPSHOT_EPOCHS:
            snapshots_valid.append(predict(model, va_loader))
            snapshots_test.append(predict(model, te_loader))

        fold_history.append(rec)
        pd.DataFrame(fold_history).to_csv(history_path, index=False)
        print(rec)

    assert len(snapshots_valid) == len(SNAPSHOT_EPOCHS)
    assert len(snapshots_test) == len(SNAPSHOT_EPOCHS)

    va_prob = np.mean(snapshots_valid, axis=0)
    te_prob = np.mean(snapshots_test, axis=0)

    assert va_prob.shape == (len(va), len(TARGET_LABELS))
    assert te_prob.shape == (len(test), len(TARGET_LABELS))
    assert np.isfinite(va_prob).all() and np.isfinite(te_prob).all()

    oof[va.index.to_numpy()] = va_prob
    test_fold_probs.append(te_prob)
    histories.extend(fold_history)

    # Atomic-ish per-fold cache: save after a fold fully succeeds.
    np.savez_compressed(
        cache_path,
        va_index=va.index.to_numpy(),
        valid=va_prob,
        test=te_prob,
    )
    print("✅ Saved fold cache:", cache_path.name)

    del model, tokenizer, tr_loader, va_loader, te_loader, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()

assert len(test_fold_probs) == N_FOLDS, f"Expected {N_FOLDS} completed folds, got {len(test_fold_probs)}"
assert np.isfinite(oof).all()

test_specialist = np.mean(test_fold_probs, axis=0)
np.savez_compressed(RUN_DIR/"specialist_predictions.npz", oof=oof, test=test_specialist)
pd.DataFrame(histories).to_csv(RUN_DIR/"training_history.csv", index=False)

print("\n✅ All 5 OOF folds completed successfully.")
print("OOF/test specialist predictions saved.")


In [ ]:

# 8. Reconstruct the current V14-family probabilities on the original 500-row validation
def logit(p):
    p = np.clip(np.asarray(p,float), 1e-6, 1-1e-6)
    return np.log(p/(1-p))
def sigmoid(z):
    return 1/(1+np.exp(-z))

label_to_i = {l:i for i,l in enumerate(ALL_LABELS)}

def v2col(frame, label):
    return frame[f"{label}_prob"].to_numpy(float)

baseline_valid = {}
baseline_test = {}
baseline_threshold = {}

# implicit_toxicity — V11
j = label_to_i["implicit_toxicity"]
baseline_valid["implicit_toxicity"] = sigmoid(
    .90*logit(v2col(v2_valid,"implicit_toxicity")) +
    .10*logit(old_bang_valid.mean(axis=0)[:,j])
)
baseline_test["implicit_toxicity"] = sigmoid(
    .90*logit(v2col(v2_test,"implicit_toxicity")) +
    .10*logit(old_bang_test.mean(axis=0)[:,j])
)
baseline_threshold["implicit_toxicity"] = 0.1416

# contains_profanity — unchanged V6/V14 family
j = label_to_i["contains_profanity"]
baseline_valid["contains_profanity"] = .95*v2col(v2_valid,"contains_profanity") + .05*old_bang_valid[0,:,j]
baseline_test["contains_profanity"] = .95*v2col(v2_test,"contains_profanity") + .05*old_bang_test[0,:,j]
baseline_threshold["contains_profanity"] = 0.245

# threatening — V14
j = label_to_i["threatening"]
baseline_valid["threatening"] = .95*v2col(v2_valid,"threatening") + .05*old_xlmr_valid[0,:,j]
baseline_test["threatening"] = .95*v2col(v2_test,"threatening") + .05*old_xlmr_test[0,:,j]
baseline_threshold["threatening"] = 0.16558668018085876

# sexual_content — unchanged V6/V14 family
baseline_valid["sexual_content"] = v2col(v2_valid,"sexual_content")
baseline_test["sexual_content"] = v2col(v2_test,"sexual_content")
baseline_threshold["sexual_content"] = 0.245

orig_valid_combined_idx = np.arange(len(train), len(train)+len(valid))
new_valid = oof[orig_valid_combined_idx]
new_test = test_specialist
new_valid_by_label = {l:new_valid[:,TARGET_LABELS.index(l)] for l in TARGET_LABELS}
new_test_by_label = {l:new_test[:,TARGET_LABELS.index(l)] for l in TARGET_LABELS}

fold_map = valid_folds.set_index("ID")["fold"]
selection_folds = valid["ID"].map(fold_map).to_numpy(int)
assert not np.isnan(selection_folds).any()

print("Current validation positive predictions:")
for l in TARGET_LABELS:
    print(l, int((baseline_valid[l]>=baseline_threshold[l]).sum()))


In [ ]:

# 9. Cross-fitted threshold + blend selection
def best_threshold(y, p):
    y = np.asarray(y,int)
    if y.sum()==0: return 1.000001
    if y.sum()==len(y): return 0.0
    precision, recall, thresholds = precision_recall_curve(y,p)
    f = 2*precision[:-1]*recall[:-1] / np.maximum(precision[:-1]+recall[:-1],1e-12)
    return float(thresholds[np.flatnonzero(np.isclose(f,f.max(),atol=1e-12,rtol=0))[-1]])

def crossfit(y, p):
    pred = np.zeros(len(y), dtype=np.int8)
    scores, thresholds = [], []
    for fold in range(N_FOLDS):
        held = selection_folds==fold
        t = best_threshold(y[~held], p[~held])
        thresholds.append(t)
        pred[held] = (p[held]>=t).astype(np.int8)
        scores.append(f1_score(y[held], pred[held], zero_division=0))
    return {
        "f1": float(f1_score(y,pred,zero_division=0)),
        "fold_scores": np.asarray(scores,float),
        "thresholds": np.asarray(thresholds,float),
        "prediction": pred,
    }

def blend_probs(a,b,w,mode):
    if mode=="prob":
        return (1-w)*a + w*b
    return sigmoid((1-w)*logit(a) + w*logit(b))

results = []
selected = {}

for label in TARGET_LABELS:
    y = valid[label].to_numpy(int)
    b = baseline_valid[label]
    n = new_valid_by_label[label]
    base_cv = crossfit(y,b)

    best = {
        "label":label,
        "mode":"unchanged",
        "weight":0.0,
        "f1":base_cv["f1"],
        "fold_scores":base_cv["fold_scores"],
        "thresholds":base_cv["thresholds"],
        "passes_gate":False,
    }

    for mode in BLEND_MODES:
        for w in BLEND_WEIGHTS[1:]:
            p = blend_probs(b,n,w,mode)
            cv = crossfit(y,p)
            wins = int((cv["fold_scores"] > base_cv["fold_scores"] + 1e-12).sum())
            losses = int((cv["fold_scores"] < base_cv["fold_scores"] - 1e-12).sum())
            ties = N_FOLDS-wins-losses
            gain = cv["f1"]-base_cv["f1"]
            passes = (
                gain >= MIN_CROSSFIT_GAIN and
                wins >= MIN_WINNING_FOLDS and
                losses <= MAX_LOSING_FOLDS
            )
            results.append({
                "label":label, "mode":mode, "weight":w,
                "crossfit_f1":cv["f1"], "baseline_crossfit_f1":base_cv["f1"],
                "gain":gain, "wins":wins, "ties":ties, "losses":losses,
                "passes_gate":passes,
                "median_threshold":float(np.median(cv["thresholds"])),
            })
            # Prefer gated candidates; then higher F1, more wins, fewer losses, smaller weight.
            key = (passes, cv["f1"], wins, -losses, -w)
            old_key = (
                best["passes_gate"], best["f1"],
                int((best["fold_scores"]>base_cv["fold_scores"]+1e-12).sum()),
                -int((best["fold_scores"]<base_cv["fold_scores"]-1e-12).sum()),
                -best["weight"],
            )
            if key > old_key:
                best = {
                    "label":label,"mode":mode,"weight":w,"f1":cv["f1"],
                    "fold_scores":cv["fold_scores"],"thresholds":cv["thresholds"],
                    "passes_gate":passes,
                }

    selected[label] = best

comparison = pd.DataFrame(results).sort_values(
    ["passes_gate","gain","wins","losses"],
    ascending=[False,False,False,True]
)
comparison.to_csv(RUN_DIR/"blend_search.csv",index=False)

summary_rows=[]
for label,best in selected.items():
    y = valid[label].to_numpy(int)
    base_cv = crossfit(y,baseline_valid[label])
    fs = best["fold_scores"]
    wins=int((fs>base_cv["fold_scores"]+1e-12).sum())
    losses=int((fs<base_cv["fold_scores"]-1e-12).sum())
    summary_rows.append({
        "label":label,
        "baseline_crossfit_f1":base_cv["f1"],
        "selected_crossfit_f1":best["f1"],
        "gain":best["f1"]-base_cv["f1"],
        "mode":best["mode"],
        "weight":best["weight"],
        "wins":wins,
        "ties":N_FOLDS-wins-losses,
        "losses":losses,
        "passes_gate":best["passes_gate"],
        "median_threshold":float(np.median(best["thresholds"])),
    })
summary = pd.DataFrame(summary_rows).sort_values("gain",ascending=False)
summary.to_csv(RUN_DIR/"selection_summary.csv",index=False)
display(summary.round(4))


In [ ]:

# 10. Build controlled submissions
# We create:
#  A) best_single: only the strongest gated label replaces V14
#  B) all_passed: every label that passed the conservative gate
# If nothing passes, both files reproduce the V14 champion.

def make_candidate(replace_labels):
    out = champion.copy()
    details = {}
    for label in replace_labels:
        best = selected[label]
        w = best["weight"]
        mode = best["mode"]
        p_test = blend_probs(baseline_test[label], new_test_by_label[label], w, mode)
        threshold = float(np.median(best["thresholds"]))
        pred = (p_test>=threshold).astype(np.int8)
        before = out[label].to_numpy(int)
        out[label] = pred
        details[label] = {
            "mode":mode,
            "weight":w,
            "threshold":threshold,
            "changed_rows":int((before!=pred).sum()),
            "positives_before":int(before.sum()),
            "positives_after":int(pred.sum()),
        }
    assert out["ID"].tolist()==champion["ID"].tolist()
    assert out[ALL_LABELS].isin([0,1]).all().all()
    return out,details

passed = summary[summary["passes_gate"]].copy()
if len(passed):
    best_label = passed.sort_values(["gain","wins"],ascending=False).iloc[0]["label"]
    best_single_labels=[best_label]
    all_passed_labels=passed["label"].tolist()
else:
    best_single_labels=[]
    all_passed_labels=[]

single, single_details = make_candidate(best_single_labels)
allp, all_details = make_candidate(all_passed_labels)

single_path = RUN_DIR/"submission_v15_best_single.csv"
all_path = RUN_DIR/"submission_v15_all_passed.csv"
single.to_csv(single_path,index=False)
allp.to_csv(all_path,index=False)

report = {
    "version":"v15_rare_label_oof_specialist",
    "champion_kaggle_score":0.586,
    "champion_file":"submission_v14_champion.csv",
    "target_labels":TARGET_LABELS,
    "best_single_labels":best_single_labels,
    "all_passed_labels":all_passed_labels,
    "single_details":single_details,
    "all_passed_details":all_details,
    "gate":{
        "min_crossfit_gain":MIN_CROSSFIT_GAIN,
        "min_winning_folds":MIN_WINNING_FOLDS,
        "max_losing_folds":MAX_LOSING_FOLDS,
    },
    "warning":"OOF/development gains do not guarantee a higher Kaggle score."
}
(RUN_DIR/"v15_report.json").write_text(json.dumps(report,indent=2))

print("\nRECOMMENDATION")
if best_single_labels:
    print("First Kaggle test:", single_path)
    print("Only changed label:", best_single_labels[0])
    print("Details:", single_details)
else:
    print("No target label passed the conservative gate.")
    print("Do NOT spend a Kaggle submission; V14 remains preferred.")

print("\nAll-passed experimental file:", all_path)


In [ ]:

# 11. Package complete results
zip_path = Path("/content/v15_results.zip")
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in RUN_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print("Created:", zip_path)
print("Send v15_results.zip back to ChatGPT for the next decision.")
from google.colab import files
files.download(str(zip_path))
